# Практическая работа №3
## «EDA и Pandas»

**Вариант 4.** Видеоигры (датасет VGChartz, 2024)

**Цель работы:** научиться проводить разведочный анализ данных (EDA) с помощью библиотеки Pandas: изучать структуру и статистические характеристики датасета, обрабатывать пропуски и дубликаты, рассчитывать коэффициенты асимметрии, эксцесса и вариации, выполнять фильтрацию, срезы, группировку и сортировку данных.

## Ход работы
### 1. EDA датасета

Подключаем библиотеку Pandas и загружаем датасет с продажами видеоигр. Каждая строка датасета — это одна игра на одной конкретной платформе.

In [1]:
import pandas as pd
df = pd.read_csv('data/vgchartz-2024.csv')
df

,img,title,console,genre,publisher,developer,critic_score,total_sales,na_sales,jp_sales,pal_sales,other_sales,release_date,last_update
0,/games/boxart/full_6510540AmericaFrontccc.jpg,Grand Theft Auto V,PS3,Action,Rockstar Games,Rockstar North,9.4,20.32,6.37,0.99,9.85,3.12,2013-09-17,NaN
1,/games/boxart/full_5563178AmericaFrontccc.jpg,Grand Theft Auto V,PS4,Action,Rockstar Games,Rockstar North,9.7,19.39,6.06,0.60,9.71,3.02,2014-11-18,2018-01-03
2,/games/boxart/827563ccc.jpg,Grand Theft Auto: Vice City,PS2,Action,Rockstar Games,Rockstar North,9.6,16.15,8.41,0.47,5.49,1.78,2002-10-28,NaN
3,/games/boxart/full_9218923AmericaFrontccc.jpg,Grand Theft Auto V,X360,Action,Rockstar Games,Rockstar North,NaN,15.86,9.06,0.06,5.33,1.42,2013-09-17,NaN
4,/games/boxart/full_4990510AmericaFrontccc.jpg,Call of Duty: Black Ops 3,PS4,Shooter,Activision,Treyarch,8.1,15.09,6.18,0.41,6.05,2.44,2015-11-06,2018-01-14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64011,/games/boxart/full_2779838AmericaFrontccc.jpg,XBlaze Lost: Memories,PC,Visual Novel,Aksys Games,Arc System Works,NaN,NaN,NaN,NaN,NaN,NaN,2016-08-11,2019-01-28
64012,/games/boxart/full_8031506AmericaFrontccc.jpg,"Yoru, Tomosu",PS4,Visual Novel,Nippon Ichi Software,Nippon Ichi Software,NaN,NaN,NaN,NaN,NaN,NaN,2020-07-30,2020-05-09
64013,/games/boxart/full_6553045AmericaFrontccc.jpg,"Yoru, Tomosu",NS,Visual Novel,Nippon Ichi Software,Nippon Ichi Software,NaN,NaN,NaN,NaN,NaN,NaN,2020-07-30,2020-05-09
64014,/games/boxart/full_6012940JapanFrontccc.png,Yunohana SpRING! ~Mellow Times~,NS,Visual Novel,Idea Factory,Otomate,NaN,NaN,NaN,NaN,NaN,NaN,2019-02-28,2019-02-24


In [2]:
df.shape

(64016, 14)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 64016 entries, 0 to 64015
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   img           64016 non-null  str    
 1   title         64016 non-null  str    
 2   console       64016 non-null  str    
 3   genre         64016 non-null  str    
 4   publisher     64016 non-null  str    
 5   developer     63999 non-null  str    
 6   critic_score  6678 non-null   float64
 7   total_sales   18922 non-null  float64
 8   na_sales      12637 non-null  float64
 9   jp_sales      6726 non-null   float64
 10  pal_sales     12824 non-null  float64
 11  other_sales   15128 non-null  float64
 12  release_date  56965 non-null  str    
 13  last_update   17879 non-null  str    
dtypes: float64(6), str(8)
memory usage: 13.6 MB


По `info()` видно, что текстовые столбцы (название, платформа, жанр, издатель, разработчик, даты) заполнены почти полностью, а числовые — оценка критиков и продажи по регионам — содержат очень много пропусков. Даты `release_date` и `last_update` хранятся как текст, а не как тип даты.

In [4]:
df.describe()

,critic_score,total_sales,na_sales,jp_sales,pal_sales,other_sales
count,6678.000000,18922.000000,12637.000000,6726.000000,12824.000000,15128.000000
mean,7.220440,0.349113,0.264740,0.102281,0.149472,0.043041
std,1.457066,0.807462,0.494787,0.168811,0.392653,0.126643
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,6.400000,0.030000,0.050000,0.020000,0.010000,0.000000
50%,7.500000,0.120000,0.120000,0.040000,0.040000,0.010000
75%,8.300000,0.340000,0.280000,0.120000,0.140000,0.030000
max,10.000000,20.320000,9.760000,2.130000,9.850000,3.120000


По `describe()`:
- оценка критиков лежит в шкале от 1 до 10, медиана немного выше среднего — низких оценок меньше, но они тянут среднее вниз;
- во всех столбцах продаж среднее заметно больше медианы, а максимум в десятки раз больше третьего квартиля: большинство игр продаются небольшими тиражами, а единичные хиты продаются огромными. Это признак сильной правой асимметрии и выбросов;
- в `other_sales` первый квартиль равен нулю: у многих игр продаж в остальных регионах мира почти нет.

In [5]:
df.nunique()

img             56177
title           39798
console            81
genre              20
publisher        3383
developer        8862
critic_score       89
total_sales       482
na_sales          320
jp_sales          121
pal_sales         256
other_sales       133
release_date     7922
last_update      1545
dtype: int64

Что показывает `nunique()`:
- уникальных названий заметно меньше, чем строк: одна и та же игра выходит на нескольких платформах и записана отдельной строкой для каждой;
- уникальных картинок `img` тоже меньше, чем строк: у части игр вместо обложки стоит общая заглушка `default.jpg`;
- платформ и жанров немного — это категориальные столбцы;
- у числовых столбцов продаж мало уникальных значений, потому что продажи округлены до сотых долей миллиона.

In [6]:
df.columns

Index(['img', 'title', 'console', 'genre', 'publisher', 'developer',
       'critic_score', 'total_sales', 'na_sales', 'jp_sales', 'pal_sales',
       'other_sales', 'release_date', 'last_update'],
      dtype='str')

Описание столбцов (по словарю датасета):

| Столбец | Что означает |
|---|---|
| `img` | ссылка на картинку обложки игры на сайте vgchartz.com |
| `title` | название игры |
| `console` | платформа (консоль), для которой вышла игра |
| `genre` | жанр игры |
| `publisher` | издатель |
| `developer` | разработчик |
| `critic_score` | оценка критиков Metacritic по шкале до 10 |
| `total_sales` | продажи копий по всему миру, млн |
| `na_sales` | продажи в Северной Америке, млн |
| `jp_sales` | продажи в Японии, млн |
| `pal_sales` | продажи в Европе и Африке, млн |
| `other_sales` | продажи в остальном мире, млн |
| `release_date` | дата выхода игры |
| `last_update` | дата последнего обновления записи |

In [7]:
col = 'total_sales'
df[col].mean()

np.float64(0.3491126730789557)

In [8]:
df[col].mode()

0    0.01
Name: total_sales, dtype: float64

In [9]:
df[col].median()

np.float64(0.12)

Для анализа выбран столбец `total_sales` — общие мировые продажи, главный показатель успеха игры для бизнеса.

Мода меньше медианы, а медиана заметно меньше среднего: чаще всего игры продаются совсем маленькими тиражами, а среднее завышено небольшим числом бестселлеров. Поэтому типичные продажи игры лучше описывает медиана, а не среднее.

In [10]:
df['console'].unique()

<ArrowStringArray>
[   'PS3',    'PS4',    'PS2',   'X360',   'XOne',     'PC',    'PSP',
    'Wii',     'PS',     'DS',   '2600',    'GBA',    'NES',     'XB',
    'PSN',    'GEN',    'PSV',     'DC',    'N64',    'SAT',   'SNES',
    'GBC',     'GC',     'NS',    '3DS',     'GB',   'WiiU',     'WS',
     'VC',     'NG',     'WW',    'SCD',    'PCE',    'XBL',    '3DO',
     'GG',    'OSX',    'Mob',   'PCFX', 'Series',    'All',    'iOS',
   '5200',    'And',   'DSiW',   'Lynx',  'Linux',     'MS',    'ZXS',
   'ACPC',   'Amig',   '7800',    'DSi',     'AJ',   'WinP',   'iQue',
    'GIZ',     'VB',   'Ouya',  'NGage',    'AST',    'MSD',   'S32X',
     'XS',    'PS5',    'Int',     'CV',    'Arc',    'C64',    'FDS',
    'MSX',     'OR',   'C128',    'CDi',   'CD32',    'BRW',    'FMT',
   'ApII',    'Aco',   'BBCM',   'TG16']
Length: 81, dtype: str

В `console` собраны платформы самых разных поколений: от старых приставок (Atari 2600, NES, SNES) до современных (PS5, Xbox Series), а также ПК, мобильные платформы (iOS, Android) и сервисы цифровой дистрибуции (PSN, XBL, VC). Отдельно стоят значения `All` и `Series` — это не консоли: `All` — сводная запись об игре сразу на всех платформах, `Series` — запись о целой серии игр (например, God of War или Frogger). Данных о продажах у таких записей нет.

In [11]:
df['genre'].unique()

<ArrowStringArray>
[          'Action',          'Shooter', 'Action-Adventure',
           'Sports',     'Role-Playing',       'Simulation',
           'Racing',            'Music',             'Misc',
         'Fighting',         'Platform',        'Adventure',
         'Strategy',           'Puzzle',              'MMO',
          'Sandbox',            'Party',        'Education',
       'Board Game',     'Visual Novel']
Length: 20, dtype: str

Жанров немного, и они хорошо подходят для группировки. `Misc` означает «прочее» — игры, которые не попали ни в один конкретный жанр.

In [12]:
df['publisher'].unique()[:20]

<ArrowStringArray>
[                     'Rockstar Games',                          'Activision',
                           'EA Sports',                     'Electronic Arts',
              'Microsoft Game Studios',                   'Microsoft Studios',
                  'Bethesda Softworks',                             'Ubisoft',
         'Sony Computer Entertainment',                      'GT Interactive',
 'Sony Computer Entertainment America',                               'Namco',
                           'LucasArts',                             'Majesco',
            'Warner Bros. Interactive',               'Universal Interactive',
                         'Square Enix',                   'Eidos Interactive',
                           'Microsoft',                           'RedOctane']
Length: 20, dtype: str

Издателей и разработчиков в датасете тысячи, поэтому полный список их уникальных значений был бы нечитаемым — выведены первые 20 издателей. Среди них крупнейшие компании индустрии: Rockstar Games, Activision, Electronic Arts, Microsoft, Sony, Ubisoft, Square Enix. Видно и то, что одна компания может быть записана по-разному (Microsoft Game Studios, Microsoft Studios и Microsoft).

In [13]:
df.isnull().mean()*100

img              0.000000
title            0.000000
console          0.000000
genre            0.000000
publisher        0.000000
developer        0.026556
critic_score    89.568233
total_sales     70.441765
na_sales        80.259623
jp_sales        89.493252
pal_sales       79.967508
other_sales     76.368408
release_date    11.014434
last_update     72.071045
dtype: float64

Больше всего пропусков в `critic_score` и `jp_sales` — почти девять из десяти строк пустые; во всех остальных столбцах продаж и в `last_update` пропусков тоже большинство. Это связано с особенностями источника:
- VGChartz отслеживает в основном продажи физических копий в рознице, поэтому у цифровых, инди- и малоизвестных игр данных о продажах просто нет;
- продажи в Японии есть только у игр, которые там официально выходили;
- критики Metacritic оценивают далеко не все игры, а в основном крупные релизы;
- `last_update` заполняется, только если запись на сайте обновлялась;
- у части старых или ещё не вышедших игр неизвестна дата выхода `release_date`.

In [14]:
df_raw = pd.read_csv('data/vgchartz-2024.csv')

Перед очисткой сохраняем исходную версию датасета в `df_raw`, чтобы потом можно было сравнить данные до и после обработки.

In [15]:
cols = ['developer', 'last_update', 'release_date']
df[cols] = df[cols].fillna('no data')

В категориальных столбцах пропуски заменены строкой `no data`. Даты `release_date` и `last_update` хранятся как текст, поэтому они тоже заполнены этой строкой: так сразу видно, что дата неизвестна, а не подставлена выдуманная.

Для числовых столбцов строка `no data` не подходит: если вписать текст в числовой столбец, он перестанет быть числовым, и с ним нельзя будет считать среднее, медиану, сумму, асимметрию и другие показатели — pandas будет выдавать ошибки или пропускать такие столбцы.

In [16]:
number_cols = ['critic_score', 'total_sales', 'na_sales', 'jp_sales', 'pal_sales', 'other_sales']
df[number_cols] = df[number_cols].fillna(df[number_cols].median())

Для числовых столбцов выбрано заполнение медианой:
- удалять строки нельзя: пропусков в этих столбцах от 70 до 90 %, и после удаления от датасета почти ничего бы не осталось;
- среднее не подходит: распределение продаж сильно скошено вправо, и среднее завышено небольшим числом бестселлеров, а медиана устойчива к таким выбросам и лучше описывает типичную игру.

Нужно учитывать, что пропусков очень много: после заполнения у большей части игр стоит одно и то же значение медианы. Это сдвигает распределение к медиане (она становится ещё и модой), поэтому результаты дальнейшего статистического анализа по этим столбцам нужно интерпретировать с осторожностью.

In [17]:
df.isnull().mean()*100

img             0.0
title           0.0
console         0.0
genre           0.0
publisher       0.0
developer       0.0
critic_score    0.0
total_sales     0.0
na_sales        0.0
jp_sales        0.0
pal_sales       0.0
other_sales     0.0
release_date    0.0
last_update     0.0
dtype: float64

Пропусков в датасете не осталось.

In [18]:
df.duplicated().sum()

np.int64(0)

Полностью совпадающих строк в датасете нет: даже если игра записана дважды, у записей отличаются ссылка на обложку, издатель, разработчик или дата.

In [19]:
df.duplicated(subset = ['title']).sum()

np.int64(24218)

Строк с повторяющимся названием очень много. Чтобы понять, ошибка это или закономерность, посмотрим на пример — Grand Theft Auto V.

In [20]:
df[df['title'] == 'Grand Theft Auto V']

,img,title,console,genre,publisher,developer,critic_score,total_sales,na_sales,jp_sales,pal_sales,other_sales,release_date,last_update
0,/games/boxart/full_6510540AmericaFrontccc.jpg,Grand Theft Auto V,PS3,Action,Rockstar Games,Rockstar North,9.4,20.32,6.37,0.99,9.85,3.12,2013-09-17,no data
1,/games/boxart/full_5563178AmericaFrontccc.jpg,Grand Theft Auto V,PS4,Action,Rockstar Games,Rockstar North,9.7,19.39,6.06,0.60,9.71,3.02,2014-11-18,2018-01-03
3,/games/boxart/full_9218923AmericaFrontccc.jpg,Grand Theft Auto V,X360,Action,Rockstar Games,Rockstar North,7.5,15.86,9.06,0.06,5.33,1.42,2013-09-17,no data
28,/games/boxart/full_8047513AmericaFrontccc.jpg,Grand Theft Auto V,XOne,Action,Rockstar Games,Rockstar North,9.0,8.72,4.70,0.01,3.25,0.76,2014-11-18,2018-04-11
18932,/games/boxart/full_9031276AmericaFrontccc.jpg,Grand Theft Auto V,PC,Action,Rockstar Games,Rockstar North,7.5,0.12,0.12,0.04,0.04,0.01,2015-04-14,2018-11-02
29011,/games/boxart/full_5096432AmericaFrontccc.png,Grand Theft Auto V,All,Action-Adventure,Rockstar Games,Rockstar North,7.5,0.12,0.12,0.04,0.04,0.01,2013-09-17,2020-10-08
29894,/games/boxart/full_8752476AmericaFrontccc.jpg,Grand Theft Auto V,PS5,Action-Adventure,Rockstar Games,Rockstar Games,7.5,0.12,0.12,0.04,0.04,0.01,2021-12-01,2020-06-11
29895,/games/boxart/full_915577AmericaFrontccc.jpg,Grand Theft Auto V,XS,Action-Adventure,Rockstar Games,Rockstar Games,7.5,0.12,0.12,0.04,0.04,0.01,2021-12-01,2020-06-12


У GTA V совпадает название, но различается платформа: игра выходила на PS3, PS4, Xbox 360, Xbox One, PC, PS5 и Xbox Series, и у каждой версии свои продажи и оценка. Это не ошибка, а закономерность — разные релизы одной игры, удалять их нельзя. Отдельно есть сводная строка `All` по всем платформам.

Ошибкой ввода можно считать только случай, когда одна и та же игра записана дважды **на одной и той же платформе**. Поэтому ищем дубликаты по паре столбцов `title` и `console`.

In [21]:
df.duplicated(subset=['title', 'console']).sum()

np.int64(225)

In [22]:
df[df.duplicated(subset=['title', 'console'], keep=False)].sort_values('title').head(10)

,img,title,console,genre,publisher,developer,critic_score,total_sales,na_sales,jp_sales,pal_sales,other_sales,release_date,last_update
32537,/games/boxart/full_absolute-chess_3AmericaFron...,Absolute Chess,DSiW,Misc,Tasuke,Tasuke,7.5,0.12,0.12,0.04,0.04,0.01,2010-08-09,no data
32538,/games/boxart/default.jpg,Absolute Chess,DSiW,Misc,Unknown,Tasuke,7.5,0.12,0.12,0.04,0.04,0.01,no data,no data
55926,/games/boxart/full_2886030AmericaFrontccc.jpg,Ace Combat 5: The Unsung War,PS2,Simulation,Namco,Namco,7.5,0.12,0.12,0.04,0.04,0.01,2004-10-25,no data
56164,/games/boxart/8720327ccc.jpg,Ace Combat 5: The Unsung War,PS2,Simulation,Namco,Project ACES,8.0,0.12,0.12,0.04,0.04,0.01,2004-10-25,no data
12460,/games/boxart/full_4008801AmericaFrontccc.jpg,Adventure Time: The Secret of the Nameless Kin...,X360,Adventure,Little Orbit,WayForward Technologies,7.5,0.06,0.00,0.04,0.05,0.00,2014-10-18,2018-04-17
17417,/games/boxart/full_2043543AmericaFrontccc.jpg,Adventure Time: The Secret of the Nameless Kin...,X360,Action-Adventure,Little Orbit,Way Forward,7.5,0.01,0.01,0.04,0.04,0.00,2014-11-18,2019-05-10
32645,/games/boxart/default.jpg,Akaneiro: Demon Hunters,PC,Misc,Spicy Horse,Unknown,7.5,0.12,0.12,0.04,0.04,0.01,2013-06-13,no data
48467,/games/boxart/full_5466711AmericaFrontccc.jpg,Akaneiro: Demon Hunters,PC,Role-Playing,Spicy Horse,Spicy Horse,7.5,0.12,0.12,0.04,0.04,0.01,2013-01-31,no data
13461,/games/boxart/2805168ccc.jpg,Akko ni Omakase! Brain Shock,DS,Misc,Taito,Taito,7.5,0.04,0.12,0.04,0.04,0.01,2006-12-07,no data
13536,/games/boxart/full_2988020JapanFrontccc.jpg,Akko ni Omakase! Brain Shock,DS,Misc,Taito,Taito Corporation,7.5,0.04,0.12,0.04,0.04,0.01,2006-12-07,no data


Найденные пары — это одна и та же игра на одной платформе, а записи различаются только мелочами: написанием названия разработчика (Way Forward и WayForward Technologies, Taito и Taito Corporation), издателем `Unknown` вместо настоящего, заглушкой вместо обложки или датой. По смыслу это одна запись, введённая в базу дважды, то есть ошибка ввода, поэтому такие дубликаты удаляем.

In [23]:
df.shape

(64016, 14)

In [24]:
df= df.drop_duplicates(subset = ['title', 'console'])

In [25]:
df.shape

(63791, 14)

In [26]:
df.duplicated(subset=['title', 'console']).sum()

np.int64(0)

После удаления исчезла лишь малая доля строк (меньше половины процента), а все разные игры и их версии на разных платформах сохранились, поэтому датасет не потерял смысл. Повторов по паре «название + платформа» не осталось.

### 2. Коэффициенты асимметрии, эксцесса и вариации

Для расчёта выбран столбец `total_sales` — мировые продажи игры, главный показатель её коммерческого успеха.

In [27]:
from scipy.stats import skew, kurtosis 

In [28]:
col = 'total_sales'

In [29]:
skew_val = skew(df[col])
kurt_val = kurtosis(df[col])
print('Skew: ', skew_val)
print('Kurt: ', kurt_val)

Skew:  15.868254250865313
Kurt:  407.4188704214554


In [30]:
ds = pd.DataFrame({'Показатель': ['Skew', 'Kurtosis', 'Ср.Знач', 'Медиана', 'Cтандартное отклонение' ], 'Значение': [skew_val, kurt_val, df[col].mean(), df[col].median(), df[col].std()]}) 
ds

,Показатель,Значение
0,Skew,15.868254
1,Kurtosis,407.418870
2,Ср.Знач,0.187926
3,Медиана,0.120000
4,Cтандартное отклонение,0.451912


**Интерпретация.** Коэффициент асимметрии намного больше нуля, а среднее больше медианы — распределение продаж сильно **правоскошенное**: у большинства игр продажи небольшие, а длинный правый хвост образуют редкие бестселлеры. Эксцесс (в `scipy` считается избыточный эксцесс, у нормального распределения он равен 0) очень большой: распределение имеет острый пик и тяжёлые хвосты, то есть в данных **много выбросов** — игр с продажами, во много раз превышающими типичные. Острый пик дополнительно усилен тем, что пропуски были заполнены одной и той же медианой.

In [31]:
cv = df[col].std()/df[col].mean() * 100
cv

np.float64(240.47323091317367)

**Интерпретация CV.** Коэффициент вариации больше 33 % означает неоднородную совокупность, а здесь стандартное отклонение даже в несколько раз превышает среднее. Продажи игр очень сильно различаются между собой, поэтому среднее значение плохо описывает «типичную» игру — для этого лучше подходит медиана.

### 3. Фильтрация, срезы, группировка и сортировка

In [32]:
q = df.query('critic_score > 9.0')
q

,img,title,console,genre,publisher,developer,critic_score,total_sales,na_sales,jp_sales,pal_sales,other_sales,release_date,last_update
0,/games/boxart/full_6510540AmericaFrontccc.jpg,Grand Theft Auto V,PS3,Action,Rockstar Games,Rockstar North,9.4,20.32,6.37,0.99,9.85,3.12,2013-09-17,no data
1,/games/boxart/full_5563178AmericaFrontccc.jpg,Grand Theft Auto V,PS4,Action,Rockstar Games,Rockstar North,9.7,19.39,6.06,0.60,9.71,3.02,2014-11-18,2018-01-03
2,/games/boxart/827563ccc.jpg,Grand Theft Auto: Vice City,PS2,Action,Rockstar Games,Rockstar North,9.6,16.15,8.41,0.47,5.49,1.78,2002-10-28,no data
7,/games/boxart/full_4653215AmericaFrontccc.jpg,Red Dead Redemption 2,PS4,Action-Adventure,Rockstar Games,Rockstar Games,9.8,13.94,5.26,0.21,6.21,2.26,2018-10-26,2018-11-02
10,/games/boxart/full_809251AmericaFrontccc.jpg,Call of Duty: Modern Warfare 2,X360,Shooter,Activision,Infinity Ward,9.5,13.53,8.54,0.08,3.63,1.28,2009-11-10,no data
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61890,/games/boxart/default.jpg,Galactic Civilizations II: Twilight of the Arnor,PC,Strategy,Stardock,Stardock,9.4,0.12,0.12,0.04,0.04,0.01,2008-04-30,no data
62769,/games/boxart/3515373ccc.jpg,Sid Meier's Alpha Centauri,PC,Strategy,Electronic Arts,Firaxis Games,9.4,0.12,0.12,0.04,0.04,0.01,1999-01-02,no data
62783,/games/boxart/full_4145610AmericaFrontccc.jpg,Sid Meier's Gettysburg!,PC,Strategy,Electronic Arts,Firaxis Games,9.3,0.12,0.12,0.04,0.04,0.01,1997-09-30,no data
62879,/games/boxart/4711963ccc.jpg,StarCraft: Brood War,PC,Strategy,Blizzard Entertainment,Blizzard Entertainment,9.1,0.12,0.12,0.04,0.04,0.01,1998-11-30,no data


Через `query()` отобраны игры с оценкой критиков выше 9 — признанные шедевры. Порог взят высокий намеренно: почти 90 % пропусков в `critic_score` заполнены медианой, и при низком пороге в выборку попали бы десятки тысяч игр без настоящей оценки. В выборке оказались известные хиты — Grand Theft Auto, Red Dead Redemption 2, Call of Duty, а также классические стратегии для PC (StarCraft, Warcraft, Sid Meier's).

In [33]:
sort = df.sort_values(by = 'total_sales', ascending = False)
sort.head(10)

,img,title,console,genre,publisher,developer,critic_score,total_sales,na_sales,jp_sales,pal_sales,other_sales,release_date,last_update
0,/games/boxart/full_6510540AmericaFrontccc.jpg,Grand Theft Auto V,PS3,Action,Rockstar Games,Rockstar North,9.4,20.32,6.37,0.99,9.85,3.12,2013-09-17,no data
1,/games/boxart/full_5563178AmericaFrontccc.jpg,Grand Theft Auto V,PS4,Action,Rockstar Games,Rockstar North,9.7,19.39,6.06,0.60,9.71,3.02,2014-11-18,2018-01-03
2,/games/boxart/827563ccc.jpg,Grand Theft Auto: Vice City,PS2,Action,Rockstar Games,Rockstar North,9.6,16.15,8.41,0.47,5.49,1.78,2002-10-28,no data
3,/games/boxart/full_9218923AmericaFrontccc.jpg,Grand Theft Auto V,X360,Action,Rockstar Games,Rockstar North,7.5,15.86,9.06,0.06,5.33,1.42,2013-09-17,no data
4,/games/boxart/full_4990510AmericaFrontccc.jpg,Call of Duty: Black Ops 3,PS4,Shooter,Activision,Treyarch,8.1,15.09,6.18,0.41,6.05,2.44,2015-11-06,2018-01-14
5,/games/boxart/full_call-of-duty-modern-warfare...,Call of Duty: Modern Warfare 3,X360,Shooter,Activision,Infinity Ward,8.7,14.82,9.07,0.13,4.29,1.33,2011-11-08,no data
6,/games/boxart/full_call-of-duty-black-ops_5Ame...,Call of Duty: Black Ops,X360,Shooter,Activision,Treyarch,8.8,14.74,9.76,0.11,3.73,1.14,2010-11-09,no data
7,/games/boxart/full_4653215AmericaFrontccc.jpg,Red Dead Redemption 2,PS4,Action-Adventure,Rockstar Games,Rockstar Games,9.8,13.94,5.26,0.21,6.21,2.26,2018-10-26,2018-11-02
8,/games/boxart/full_1977964AmericaFrontccc.jpg,Call of Duty: Black Ops II,X360,Shooter,Activision,Treyarch,8.4,13.86,8.27,0.07,4.32,1.20,2012-11-13,2018-04-07
9,/games/boxart/full_4649679AmericaFrontccc.png,Call of Duty: Black Ops II,PS3,Shooter,Activision,Treyarch,8.0,13.80,4.99,0.65,5.88,2.28,2012-11-13,2018-04-07


Через `sort_values()` датасет отсортирован по мировым продажам `total_sales` по убыванию. Наверху оказались самые продаваемые игры: GTA V сразу на нескольких платформах, Grand Theft Auto: Vice City, несколько частей Call of Duty и Red Dead Redemption 2. Почти все они выпущены двумя издателями — Rockstar Games и Activision — и относятся к жанрам экшен и шутер.

In [34]:
df.loc[0:5, ['critic_score', 'last_update']]

,critic_score,last_update
0,9.4,no data
1,9.7,2018-01-03
2,9.6,no data
3,7.5,no data
4,8.1,2018-01-14
5,8.7,no data


Срез через `loc` выбирает строки и столбцы по меткам: строки с индексом от 0 до 5 (в `loc` правая граница включается) и столбцы `critic_score` и `last_update` по названиям.

In [35]:
df.loc[df['publisher'] == 'Konami', ['title', 'critic_score', 'total_sales']].head(10)

,title,critic_score,total_sales
118,Winning Eleven: Pro Evolution Soccer 2007,8.8,4.39
149,World Soccer Winning Eleven 9,8.9,4.06
163,World Soccer Winning Eleven 8 International,9.3,3.85
176,Pro Evolution Soccer 2008,8.2,3.63
263,World Soccer Winning Eleven 6 International,7.5,2.99
355,PES 2009: Pro Evolution Soccer,7.5,2.50
380,Pro Evolution Soccer 2011,6.0,2.41
390,Frogger's Adventures: Temple of the Frog,6.7,2.39
418,Pro Evolution Soccer 2010,7.0,2.28
431,Dance Dance Revolution X2,7.5,2.23


Здесь выбраны определённые столбцы (название, оценка, продажи) только для игр издателя Konami. Верх списка занимают футбольные симуляторы Pro Evolution Soccer / Winning Eleven — самая успешная серия этого издателя.

In [36]:
df.iloc[0:5, 4:5]

,publisher
0,Rockstar Games
1,Rockstar Games
2,Rockstar Games
3,Rockstar Games
4,Activision


Срез через `iloc` выбирает данные по номерам позиций, а не по названиям: первые пять строк и столбцы по их порядковым номерам. В отличие от `loc`, правая граница в `iloc` не включается.

In [37]:
df.groupby('genre')['total_sales'].sum().sort_values(ascending=False)

genre
Action              1807.85
Sports              1540.38
Shooter             1464.96
Misc                1429.17
Role-Playing         931.65
Adventure            847.76
Racing               763.83
Platform             713.22
Simulation           544.31
Fighting             520.17
Puzzle               461.26
Strategy             460.40
Action-Adventure     341.93
Music                 69.75
Visual Novel          38.42
Party                 20.85
MMO                   19.39
Education              4.60
Sandbox                4.17
Board Game             3.93
Name: total_sales, dtype: float64

Данные сгруппированы по жанру, и для каждого жанра посчитана сумма мировых продаж `total_sales`, результат отсортирован по убыванию. Больше всего копий продано у экшенов, спортивных игр и шутеров — это самые массовые и коммерчески успешные жанры, к ним относятся крупнейшие серии вроде GTA, FIFA и Call of Duty. Меньше всего продаж у нишевых жанров: настольных и обучающих игр, песочниц, MMO и визуальных новелл.

Нужно учитывать, что пропуски в продажах заполнены медианой, поэтому жанры с большим количеством игр без данных получают «добавку». Сильнее всего это заметно у `Misc`: больше половины его суммы составляют подставленные значения, и по реальным продажам он был бы ниже. Тройка лидеров при этом не меняется и по реальным данным.

### 4. Топ-5 записей

In [38]:
top5 = df.sort_values(by='total_sales', ascending=False).drop_duplicates(subset = ['title']).head(5)[['title', 'publisher', 'total_sales']]
top5

,title,publisher,total_sales
0,Grand Theft Auto V,Rockstar Games,20.32
2,Grand Theft Auto: Vice City,Rockstar Games,16.15
4,Call of Duty: Black Ops 3,Activision,15.09
5,Call of Duty: Modern Warfare 3,Activision,14.82
6,Call of Duty: Black Ops,Activision,14.74


Топ-5 построен по признаку `total_sales` — мировым продажам копий. Это главный показатель коммерческого успеха игры: именно продажи приносят издателю деньги, поэтому для бизнеса этот признак значимее оценки критиков. Кроме того, `critic_score` почти на 90 % заполнен медианой, и рейтинг по нему был бы ненадёжным.

Перед выбором пятёрки датасет отсортирован по убыванию продаж, а затем из него убраны повторы названий: одна и та же игра выходила на разных платформах, и без этого GTA V заняла бы в рейтинге сразу три места. Для каждой игры осталась её самая продаваемая версия, поэтому топ-5 показывает пять разных самых успешных игр.

В рейтинг попали только игры двух издателей — Rockstar Games (Grand Theft Auto) и Activision (Call of Duty). На это повлияло то, что:
- это продолжения культовых серий с огромной аудиторией фанатов и большими маркетинговыми бюджетами;
- игры выходили сразу на нескольких популярных платформах;
- большинство из них вышли во времена PS2, PS3 и Xbox 360, когда игры в основном покупали на физических дисках, а VGChartz учитывает именно такие продажи — у новых игр, которые чаще покупают цифровыми копиями, продажи в датасете занижены.

## Вывод
В ходе выполнения данной практической работы научились проводить разведочный анализ данных (EDA) с помощью библиотеки Pandas: изучать структуру и статистические характеристики датасета, обрабатывать пропуски и дубликаты, рассчитывать коэффициенты асимметрии, эксцесса и вариации, выполнять фильтрацию, срезы, группировку и сортировку данных.